In [0]:
%sql
-- 1. Create a new Unity Catalog catalog
CREATE CATALOG IF NOT EXISTS my_catalog 
COMMENT 'This is my custom catalog';

-- 2. Create a new schema (database) inside that catalog
CREATE SCHEMA IF NOT EXISTS my_catalog.my_schema 
COMMENT 'This is my custom schema';

use catalog my_catalog;
use schema my_schema;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when

# 1. Initialize Spark Session with sufficient driver memory
spark = SparkSession.builder \
    .appName("ScaleTo1Billion") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# 2. Generate 1 billion rows natively in Spark (0 to 999,999,999)
# This creates a DataFrame with a single column named "id"
total_rows = 1_000_000_000
country_df = spark.range(0, total_rows)

# 3. Rename "id" to "order_id" and dynamically assign countries based on a distribution pattern
# Using modulo (%) allows us to cleanly distribute rows into percentage buckets without loops
country_df = country_df.withColumn("order_id", col("id") + 1) \
    .withColumn(
        "country",
        when(expr("id % 100 < 90"), "US")         # 90% US
        .when(expr("id % 100 < 95"), "India")     # 5% India
        .when(expr("id % 100 < 98"), "UK")        # 3% UK
        .when(expr("id % 100 < 100"), "Canada")   # 2% Canada
        .otherwise("Germany")                     # Fallback / Extra Country
    ).drop("id") # Drop the original 'id' column

# 4. (Optional) Verify your exact distribution count across 1 billion rows
# Warning: .count() on 1B rows takes some time depending on your cluster size
# country_df.groupBy("country").count().show()


In [0]:
country_df.show()

In [0]:
country_df.count()

In [0]:
country_df.groupBy('country').count().collect() 

In [0]:
from pyspark.sql.functions import count,desc

country_df.groupBy('country').agg(count(col('order_id')).alias('count')).orderBy(desc('count')).show()


In [0]:
lookup = [
    ("US", "North America"),
    ("India", "Asia"),
    ("UK", "Europe"),
    ("Canada", "North America")
]

c_df = spark.createDataFrame(
    lookup,
    ["country","continent"]
)

In [0]:
c_df.show()

This is taking very long

In [0]:
country_df.join(c_df,'country').show()

In [0]:
country_df.join(c_df,'country').explain("formatted")

## USE BROADCAST JOINS

In [0]:
from pyspark.sql.functions import broadcast
country_df.join(broadcast(c_df),'country').explain()

In [0]:
from pyspark.sql.functions import broadcast
country_df.join(broadcast(c_df),'country').display()

In [0]:
result = country_df.join(broadcast(c_df), "country")

In [0]:
result.explain(True)

## SALTED

Add Salt to Large Dataset

In [0]:
country_df.select('country').distinct().show()

In [0]:
from pyspark.sql.functions import floor, rand

country_df_salted = country_df.withColumn("salt",floor(rand() * 10))

In [0]:
country_df_salted.show(3)

In [0]:
from pyspark.sql.functions import concat_ws

country_df_salted = country_df_salted.withColumn("country_salt", concat_ws("_", col("country"), col("salt")))

In [0]:
country_df_salted.show(3,False)

In [0]:
country_lookup = spark.createDataFrame(
    [
        ("US", "North America"),
        ("India", "Asia"),
        ("UK", "Europe"),
        ("Canada", "North America")
    ],
    ["country", "region"]
)

In [0]:
country_lookup.show(3)

In [0]:
salts = spark.range(10).withColumnRenamed("id", "salt")

In [0]:
lookup_expanded = country_lookup.crossJoin(salts)

In [0]:
lookup_expanded.show(3)

In [0]:
lookup_expanded = lookup_expanded.withColumn("country_salt",concat_ws("_", col("country"), col("salt")))

In [0]:
lookup_expanded.display()

In [0]:
country_df_salted.join(lookup_expanded,"country_salt").show()